<table style="width:100%">
  <tr>
    <td valign="top"><img src="https://github.com/mgrubisic/ai_bootcamp_foundations/blob/main/data/img/FER_logo_2.png?raw=1" width=300 height=80 align="left"></td>
    <td valign="top"><img src="https://github.com/mgrubisic/ai_bootcamp_foundations/blob/main/data/img/LARES_2_transparent.png?raw=1" width=250 height=80 align="right"></td>
  </tr>
 </table>

# House Prices - Advanced Regression Techniques
Predict sales prices and practice the basic concepts of machine learning...
Our competition with dataset, competition rules and other information can be found on [Kaggle competition](https://www.kaggle.com/t/2d8eb2d88f9090f65257fb7975163ae5)

The original dataset and many exemplary notebooks can be found on [Kaggle original dataset](https://www.kaggle.com/c/house-prices-advanced-regression-techniques).

![title](https://github.com/mgrubisic/ai_bootcamp_foundations/blob/main/data/img/house_prices.jpg?raw=1)

## Competition

1. Create your own [Kaggle profile](https://www.kaggle.com/account/login?phase=startRegisterTab&returnUrl=%2F),
2. Join our "AI Bootcamp house prices" competition using the
[invitation link](https://www.kaggle.com/t/2d8eb2d88f9090f65257fb7975163ae5)
3. Analyse the [dataset](https://www.kaggle.com/competitions/ai-bootcamp-house-prices/data) and [competition rules](https://www.kaggle.com/competitions/ai-bootcamp-house-prices/rules),
4. Take special care with [metrics](https://www.kaggle.com/c/ai-bootcamp-house-prices/overview/evaluation),
5. Check out publicly shared [notebooks](https://www.kaggle.com/c/house-prices-advanced-regression-techniques/code?competitionId=5407&sortBy=voteCount)  with solutions using the same dataset (public version),
6. Tune models and generate predictions,
7. Generate a [submission.csv file](https://www.kaggle.com/c/ai-bootcamp-house-prices/data?select=sample_submission.csv) and submit it for grading using the Submit Predictions button in the upper-right corner of the competition page.

Have fun and good luck!

## Part 0 - Setup

Run this cell first, in every notebook. It fetches the course repository into
the Colab session and moves into the `notebooks/` folder, so that the
`../data/...` paths work.

It is safe to run more than once, and safe after a restart.

Note: outside Colab the cell does nothing except report the working directory.
Start Jupyter from inside `notebooks/` and the paths work the same way.

If it prints `data ok: True`, you are set.

In [2]:
# --- SETUP: run this first ---
# works in Colab and locally, safe to run more than once
import os, sys, subprocess
REPO = "ai_bootcamp_foundations"
if "google.colab" in sys.modules:
    if not os.path.isdir(f"/content/{REPO}"):
        subprocess.run(["git", "clone", "-q",
                        f"https://github.com/mgrubisic/{REPO}.git"],
                       cwd="/content", check=True)
    os.chdir(f"/content/{REPO}/notebooks")
print("cwd:", os.getcwd(), "| data ok:", os.path.isdir("../data"))

cwd: /content/ai_bootcamp_foundations/notebooks | data ok: True


## Import libraries

In [3]:
import pandas as pd
import numpy as np

In [4]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

## Load datasets

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_log_error
import lightgbm as lgb

In [6]:
# import training data
df = pd.read_csv('../data/housing_prices/Kaggle/train.csv', index_col='Id')
# remove features with missing data
total = df.isnull().sum().sort_values(ascending=False)
percent = (df.isnull().sum() / df.isnull().count() * 100).sort_values(ascending=False)
missing_data = pd.concat([total, percent], axis=1, keys=['Total', 'Percent'])
df = df.drop((missing_data[missing_data['Percent'] > 0.5]).index, axis=1)
df = df.drop(df.loc[df['Electrical'].isnull()].index)
# use only the most correlated features
corr = df.corr(numeric_only=True)
relevant_inp = np.abs(corr['SalePrice']).sort_values(ascending=False).index[0:11]
# separate X and y and split to train and validation sets
X = df.loc[df.index, relevant_inp[1:]].apply(np.log1p)  # logarithm transformation
y = np.log1p(df.loc[df.index, relevant_inp[0]])  # logarithm transformation
# separate train and validation datasets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=1)

In [7]:
# Train a simple LightGBM model
model = lgb.LGBMRegressor(random_state=42, n_estimators=100, verbose=-1)
model.fit(X_train, y_train)

# TODO: Can you improve this model?
# Try different hyperparameters or use GridSearchCV.

LGBMRegressor(random_state=42, verbose=-1)

In [8]:
# Generate predictions and calculate RMSLE
y_pred = model.predict(X_val)

# Convert back from log scale for evaluation
y_val_original = np.expm1(y_val)
y_pred_original = np.expm1(y_pred)

# Calculate RMSLE
rmsle = np.sqrt(mean_squared_log_error(y_val_original, y_pred_original))
print('RMSLE: %.4f' % rmsle)

RMSLE: 0.1682


In [9]:
# Train final model on all data
model_final = lgb.LGBMRegressor(random_state=42, n_estimators=100, verbose=-1)
model_final.fit(X, y)

LGBMRegressor(random_state=42, verbose=-1)

In [10]:
# Import test dataset
X_test = pd.read_csv('../data/housing_prices/Kaggle/test.csv', index_col='Id')
X_test = X_test.loc[:,relevant_inp[1:]].apply(np.log1p)  # logarithm transformation
X_test

,OverallQual,GrLivArea,GarageCars,GarageArea,TotalBsmtSF,1stFlrSF,FullBath,TotRmsAbvGrd,YearBuilt,YearRemodAdd
Id,,,,,,,,,,
1461,1.791759,6.799056,0.693147,6.594413,6.783325,6.799056,0.693147,1.791759,7.581720,7.581720
1462,1.945910,7.192934,0.693147,5.746203,7.192934,7.192934,0.693147,1.945910,7.580189,7.580189
1463,1.791759,7.396335,1.098612,6.180017,6.834109,6.834109,1.098612,1.945910,7.599902,7.600402
1464,1.945910,7.380879,1.098612,6.154858,6.831954,6.831954,1.098612,2.079442,7.600402,7.600402
1465,2.197225,7.155396,1.098612,6.228511,7.155396,7.155396,1.098612,1.791759,7.597396,7.597396
...,...,...,...,...,...,...,...,...,...,...
2915,1.609438,6.996681,0.000000,0.000000,6.304449,6.304449,0.693147,1.791759,7.586296,7.586296
2916,1.609438,6.996681,0.693147,5.659482,6.304449,6.304449,0.693147,1.945910,7.586296,7.586296
2917,1.791759,7.110696,1.098612,6.357842,7.110696,7.110696,0.693147,2.079442,7.581210,7.599401


In [11]:
# Generate predictions on test data
y_pred_log = model_final.predict(X_test)
y_pred = np.expm1(y_pred_log)  # Convert back from log scale

## Create submission csv file
Take y_pred object (array of floats) and create a submission CSV file for uploading to Kaggle.

In [12]:
df_pred = pd.DataFrame(y_pred)
s_sub = pd.read_csv("../data/housing_prices/Kaggle/sample_submission.csv")
sub_df = pd.concat([s_sub['Id'], df_pred], axis=1)
sub_df.columns=['Id', 'SalePrice']
sub_df.to_csv("../data/housing_prices/Kaggle/TEST_submission.csv", index=False)

In [14]:
pd.read_csv("../data/housing_prices/Kaggle/TEST_submission.csv")

,Id,SalePrice
0,1461,126341.368626
1,1462,147301.886373
2,1463,158370.788072
3,1464,182425.808517
4,1465,196730.612649
...,...,...
1454,2915,73637.932761
1455,2916,78667.169916
1456,2917,151817.357371
1457,2918,106297.331791


### Isprobavanje različitih modela i Ansambla
U sljedećim ćelijama uvozimo potrebne biblioteke i pripremamo rječnik sa svim traženim modelima.

In [29]:
# Instalacija nedostajuce biblioteke
!pip install catboost -q

# Uvoz potrebnih biblioteka za sve modele
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor, GradientBoostingRegressor, VotingRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
import lightgbm as lgb
import numpy as np
from sklearn.metrics import mean_squared_log_error
import pandas as pd

In [30]:
# Inicijalizacija svih modela sa zadanim (default) postavkama
models_dict = {
    "LinearnaRegresija": LinearRegression(),
    "SlucajnaSuma": RandomForestRegressor(random_state=42),
    "AdaBoost": AdaBoostRegressor(random_state=42),
    "GBDT": GradientBoostingRegressor(random_state=42),
    "XGBoost": XGBRegressor(random_state=42),
    "LightGBM": lgb.LGBMRegressor(random_state=42, verbose=-1),
    "CatBoost": CatBoostRegressor(random_state=42, verbose=0) # verbose=0 sprječava ispisivanje previše teksta tijekom treniranja
}

results_dict = {}

In [31]:
# Treniranje i evaluacija svakog modela na validacijskom skupu
print("--- Evaluacija pojedinačnih modela ---")
for model_name, model in models_dict.items():
    # Treniranje modela na trening skupu
    model.fit(X_train, y_train)

    # Predikcija na validacijskom skupu
    val_predictions_log = model.predict(X_val)

    # Vraćanje iz logaritamske skale u stvarne cijene
    y_val_orig = np.expm1(y_val)
    val_predictions_orig = np.expm1(val_predictions_log)

    # Postavljanje negativnih predikcija na nulu (kako bi izbjegli greške u RMSLE)
    val_predictions_orig = np.maximum(val_predictions_orig, 0)

    # Izračun RMSLE metrike
    rmsle_score = np.sqrt(mean_squared_log_error(y_val_orig, val_predictions_orig))
    results_dict[model_name] = rmsle_score
    print(f"{model_name} RMSLE: {rmsle_score:.4f}")

--- Evaluacija pojedinačnih modela ---
LinearnaRegresija RMSLE: 0.1408
SlucajnaSuma RMSLE: 0.1457
AdaBoost RMSLE: 0.1879
GBDT RMSLE: 0.1324
XGBoost RMSLE: 0.1478
LightGBM RMSLE: 0.1396
CatBoost RMSLE: 0.1282


In [32]:
# Izrada 'Ansambla' (Ensemble) kombiniranjem najboljih modela temeljenih na stablima odlučivanja
# VotingRegressor uzima predikcije svih navedenih modela i računa prosjek
ensemble_estimators = [
    ('rf', models_dict["SlucajnaSuma"]),
    ('gbdt', models_dict["GBDT"]),
    ('xgb', models_dict["XGBoost"]),
    ('lgb', models_dict["LightGBM"]),
    ('cat', models_dict["CatBoost"])
]

ensemble_model = VotingRegressor(estimators=ensemble_estimators)

# Treniranje i evaluacija Ansambla
print("\n--- Evaluacija Ansambla ---")
ensemble_model.fit(X_train, y_train)
ensemble_pred_log = ensemble_model.predict(X_val)
ensemble_pred_orig = np.expm1(ensemble_pred_log)
ensemble_pred_orig = np.maximum(ensemble_pred_orig, 0)

ensemble_rmsle = np.sqrt(mean_squared_log_error(np.expm1(y_val), ensemble_pred_orig))
results_dict["Ansambl"] = ensemble_rmsle
models_dict["Ansambl"] = ensemble_model

print(f"Ansambl RMSLE: {ensemble_rmsle:.4f}")


--- Evaluacija Ansambla ---
Ansambl RMSLE: 0.1331


In [33]:
# Pronalazak najboljeg modela s najmanjom RMSLE greškom
best_model_name = min(results_dict, key=results_dict.get)
best_model_rmsle = results_dict[best_model_name]

print(f"\n🏆 Najbolji model na validacijskom skupu je: {best_model_name} s RMSLE = {best_model_rmsle:.4f}")

# Priprema najboljeg modela za konačnu predaju na Kaggle
print("Treniranje najboljeg modela na cijelom dostupnom skupu (X, y) za maksimalnu preciznost...")
best_model = models_dict[best_model_name]
best_model.fit(X, y) # Treniramo na X i y koje si definirao ranije (bez train/val podjele)

# Predikcija na Kaggle testnom skupu
test_pred_log = best_model.predict(X_test)
test_pred_orig = np.expm1(test_pred_log)

# Kreiranje DataFrame-a i spremanje u CSV za Kaggle submission
s_sub = pd.read_csv("../data/housing_prices/Kaggle/sample_submission.csv")
final_sub_df = pd.DataFrame({
    'Id': s_sub['Id'],
    'SalePrice': test_pred_orig
})

# Spremanje rezultata
final_sub_df.to_csv("../data/housing_prices/Kaggle/BEST_submission.csv", index=False)
print("✅ Datoteka 'BEST_submission.csv' je uspješno spremljena i spremna za upload na Kaggle!")
display(final_sub_df.head())


🏆 Najbolji model na validacijskom skupu je: CatBoost s RMSLE = 0.1282
Treniranje najboljeg modela na cijelom dostupnom skupu (X, y) za maksimalnu preciznost...
✅ Datoteka 'BEST_submission.csv' je uspješno spremljena i spremna za upload na Kaggle!


,Id,SalePrice
0,1461,126806.925610
1,1462,163382.950517
2,1463,187946.853542
3,1464,194476.222557
4,1465,179658.404219


### Napredna priprema podataka: One-Hot Encoding svih kategoričkih varijabli

In [28]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# 1. Ponovno učitavanje originalnih podataka
train_df = pd.read_csv('../data/housing_prices/Kaggle/train.csv', index_col='Id')
test_df = pd.read_csv('../data/housing_prices/Kaggle/test.csv', index_col='Id')

# Odvajanje ciljne varijable i logaritamska transformacija
y = np.log1p(train_df['SalePrice'])
train_df.drop('SalePrice', axis=1, inplace=True)

# Spajanje train i test skupa radi lakšeg i dosljednog One-Hot Encodinga
all_data = pd.concat([train_df, test_df], axis=0)

# 2. Rješavanje nedostajućih vrijednosti (Imputacija)
cat_cols = all_data.select_dtypes(include=['object']).columns
num_cols = all_data.select_dtypes(exclude=['object']).columns

# Kategoričke varijable popunjavamo tekstom "None"
all_data[cat_cols] = all_data[cat_cols].fillna('None')
# Numeričke varijable popunjavamo medijanom te kolone
for col in num_cols:
    all_data[col] = all_data[col].fillna(all_data[col].median())

# 3. Logaritamska transformacija za asimetrične (skewed) numeričke varijable
for col in num_cols:
    if all_data[col].skew() > 0.75:
        all_data[col] = np.log1p(all_data[col])

# 4. One-Hot Encoding za sve kategoričke varijable
# get_dummies pretvara tekstualne kategorije u više stupaca s 0 i 1 (ili True/False)
all_data = pd.get_dummies(all_data, columns=cat_cols)

# 5. Vraćanje u originalne skupove (X i X_test)
X = all_data.iloc[:len(train_df), :]
X_test = all_data.iloc[len(train_df):, :]

# 6. Podjela na train i validation skup
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Stari broj značajki: {len(num_cols)}")
print(f"Novi broj značajki nakon One-Hot Encodinga: {X.shape[1]}")
print("Podaci su uspješno pripremljeni! Sada možete ponovno pokrenuti ćelije za treniranje modela.")

Stari broj značajki: 36
Novi broj značajki nakon One-Hot Encodinga: 310
Podaci su uspješno pripremljeni! Sada možete ponovno pokrenuti ćelije za treniranje modela.



_Course: AI Bootcamp: Foundations of AI_  
_Notebook: 8_Kaggle_  
_Kaggle Competition: [AI Bootcamp House Prices](https://www.kaggle.com/competitions/ai-bootcamp-house-prices)_

_University of Zagreb Faculty of Electrical Engineering and Computing_  
_Laboratory for Renewable Energy Systems_  
_Website: [www.lares.fer.hr](https://www.lares.fer.hr/)_  
_Contact: [Hrvoje Novak](mailto:hrvoje.novak@fer.hr)_


